# Lesson 07 - 规划设计模式

本笔记本演示了使用微软代理框架的 **规划设计模式** 用于 AI 代理。
您将学习如何将复杂的旅行请求拆分为结构化的子任务，
将它们分配给专门的代理，并执行生成的计划——这一切都使用由 Pydantic 模型驱动的结构化输出完成。


## 设置


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 任务分解

任务分解是规划设计模式的核心。我们不是让单个代理端到端地处理复杂请求，而是将问题拆分为更小、定义明确的**子任务**。每个子任务分配给专门的代理（例如，航班、酒店、活动），并明确优先级和依赖顺序。

这种方法带来了几个好处：
- **清晰性**：每个子任务都有单一职责。
- **并行性**：独立的子任务可以并发运行。
- **可靠性**：失败被隔离到单个子任务。
- **预算跟踪**：成本按子任务估算并汇总。


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## 创建具有结构化输出的规划代理

规划代理充当**前台协调员**。根据高级的旅行请求，它生成一个结构化的`TravelPlan`——将请求分解为子任务，设定优先级，并识别依赖关系，以便礼宾或执行层能够完成工作。


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## 使用专用工具执行计划

一旦前台代理制定了结构化计划，**礼宾代理**便会执行它。  
每个专用工具处理一类子任务（航班、酒店、活动）。礼宾代理按照依赖顺序遍历计划中的子任务，并将每个子任务分派给相应的工具。


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## 摘要

在本课中，您学习了 AI 代理的 **规划设计模式**：

1. **任务分解** — 前台规划代理使用 Pydantic 模型将复杂的旅行请求拆分为结构化的子任务，分配给具有优先级和依赖关系的专家代理。
2. **结构化输出** — 通过传递 `response_format`，代理返回经过验证的 `TravelPlan` 对象，而不是自由格式文本，使下游处理更加可靠。
3. **计划执行** — 礼宾代理使用专家工具（`book_flight`、`reserve_hotel`、`book_activity`）迭代子任务，执行计划并报告结果。

此模式将“做什么”（规划）与“如何做”（执行）分离，使代理更加模块化、可测试且更易扩展。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：  
本文档使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保准确性，但请注意自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于重要信息，建议采用专业人工翻译。因使用本翻译造成的任何误解或曲解，我们不承担任何责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
